# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |

> **Works on:** Google Colab, Kaggle, and local Jupyter environments.
> PyTorch index (CUDA/CPU) and system dependencies are auto-detected.

### Supported languages

Chinese, English, French, German, Italian, Japanese, Korean, Portuguese, Russian, Spanish

### Voice options

- **Voice Themes** — 16 pre-defined voice styles (narrator, young, deep, warm, news, storyteller, kid, teen — male & female). No files needed!
- **Preset Voices** — Clone from pre-built HuggingFace profiles
- **Custom Voice** — Upload your own 10-30 sec audio clip
- **Auto-Clone** — Automatically clone the original speaker's voice from the source audio. No files or settings needed!

In [ ]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

import subprocess, sys, shutil, os, time

# ── Detect runtime environment ───────────────────────────────────
IN_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

ENV_LABEL = "Google Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"
print(f"🔍 Detected environment: {ENV_LABEL}")

def _pip_install(*args, **kwargs):
    """Run pip install with sys.executable for portability."""
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    subprocess.check_call(cmd, **kwargs)

# ── Ensure pip is available ──────────────────────────────────────
try:
    subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    print("📥 Bootstrapping pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"],
                          stdout=subprocess.DEVNULL)

# ── Detect CUDA availability & pick the right PyTorch index ──────
def _detect_torch_index():
    """Return (index_url, cuda_tag) based on available hardware."""
    # Check if a compatible torch is already installed
    # NOTE: use importlib.metadata instead of `import torch` to avoid
    # locking a stale version in sys.modules before pip can upgrade it.
    try:
        from importlib.metadata import version as _meta_version
        ver = _meta_version("torch")   # e.g. "2.8.0+cu128"
        if "+cu" in ver:
            tag = ver.split("+")[1]     # e.g. "cu128"
            return f"https://download.pytorch.org/whl/{tag}", tag
        elif "+cpu" in ver:
            return "https://download.pytorch.org/whl/cpu", "cpu"
    except Exception:
        pass

    # No torch installed — probe for CUDA via nvidia-smi
    try:
        out = subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL, text=True)
        # Parse CUDA version from nvidia-smi output
        for line in out.splitlines():
            if "CUDA Version" in line:
                cuda_ver = line.split("CUDA Version:")[1].strip().split()[0]
                major, minor = cuda_ver.split(".")[:2]
                tag = f"cu{major}{minor}"
                return f"https://download.pytorch.org/whl/{tag}", tag
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass

    # No GPU detected — use CPU build
    return "https://download.pytorch.org/whl/cpu", "cpu"

TORCH_INDEX, CUDA_TAG = _detect_torch_index()
print(f"🔧 PyTorch index: {CUDA_TAG}")

# ── PyTorch — install/upgrade to matching versions ───────────────
print("Installing PyTorch...")
_pip_install("torch", "torchaudio", "torchvision",
             "--index-url", TORCH_INDEX)

# Verify torch + torchaudio are ABI-compatible (use pip metadata — never reload torch)
from importlib.metadata import version as _pkg_version
_tv = _pkg_version("torch").split("+")[0]       # e.g. "2.8.0"
_av = _pkg_version("torchaudio").split("+")[0]
_vv = _pkg_version("torchvision").split("+")[0] # e.g. "0.23.0"
print(f"   torch={_pkg_version('torch')}  torchaudio={_pkg_version('torchaudio')}  torchvision={_pkg_version('torchvision')}")

if _tv.rsplit(".", 1)[0] != _av.rsplit(".", 1)[0]:
    print(f"⚠️  Version mismatch (torch {_tv} vs torchaudio {_av}). Reinstalling torchaudio…")
    _pip_install(f"torchaudio=={_tv}", "--index-url", TORCH_INDEX, "--force-reinstall", "--no-deps")

# ── Mazinger + Gradio ────────────────────────────────────────────
print("Installing Mazinger with Qwen TTS + Faster Whisper...")
_pip_install("mazinger[tts,transcribe-faster]", "gradio>=4.0")

# ── ffmpeg (required for audio extraction) ───────────────────────
if not shutil.which("ffmpeg"):
    print("📥 Installing ffmpeg...")
    if IN_COLAB or os.path.exists("/usr/bin/apt-get"):
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    elif shutil.which("conda"):
        subprocess.check_call(["conda", "install", "-y", "-q", "-c", "conda-forge", "ffmpeg"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        print("⚠️  Could not install ffmpeg automatically. Please install it manually.")
else:
    print("✅ ffmpeg available")

print("\n✅ Dependencies installed! Run the next cell to download models.")

In [ ]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS: ~3-4 GB

import subprocess, sys, shutil, time, os
import base64, io

# ── Configuration ────────────────────────────────────────────────
OLLAMA_MODEL = "translategemma:4b" # "qwen3.5:4b-q8_0"       # ← change this to use a different model
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL  # propagate to Gradio app

# ── Helper: detect environment (re-detect in case Step 1 was skipped) ─
IN_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
_HAS_APT  = os.path.exists("/usr/bin/apt-get")

# ── Helper: run a command with optional sudo ─────────────────────
def _maybe_sudo(*cmd, **kw):
    """Prepend 'sudo' only when available and needed (not root already)."""
    if os.geteuid() != 0 and shutil.which("sudo"):
        return subprocess.check_call(["sudo"] + list(cmd), **kw)
    return subprocess.check_call(list(cmd), **kw)

# ── Helper: wait for Ollama server readiness ─────────────────────
def _wait_for_ollama(url="http://localhost:11434/api/tags", retries=15, delay=2):
    """Poll Ollama until it responds (up to retries * delay seconds)."""
    import urllib.request, urllib.error
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return True
        except (urllib.error.URLError, OSError):
            pass
        time.sleep(delay)
    return False

def _ollama_is_running():
    """Check if Ollama is already serving."""
    import urllib.request, urllib.error
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
            return r.status == 200
    except (urllib.error.URLError, OSError):
        return False

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    if _HAS_APT:
        _maybe_sudo("apt-get", "install", "-y", "-qq", "zstd",
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | bash"],
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background only if not already running
if not _ollama_is_running():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("⏳ Waiting for Ollama server to be ready...")
    if _wait_for_ollama():
        print("✅ Ollama server is ready")
    else:
        print("⚠️  Ollama server did not become ready in time — pull may fail")
else:
    print("✅ Ollama server already running")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print(f"1️⃣  Pulling Ollama LLM model ({OLLAMA_MODEL})...")
_ollama_ok = False
for _attempt in range(3):
    result = subprocess.run(
        ["ollama", "pull", OLLAMA_MODEL],
        timeout=600,
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        _ollama_ok = True
        print("   ✅ Ollama model ready\n")
        break
    else:
        _err = (result.stderr or result.stdout or "").strip()
        print(f"   ⚠️  Pull attempt {_attempt + 1}/3 failed: {_err}")
        time.sleep(3)

if not _ollama_ok:
    print("   ❌ Ollama model pull failed after 3 attempts.")
    print(f"      Try running manually: !ollama pull {OLLAMA_MODEL}\n")

# ── 2. Faster Whisper model (large-v3) ──────────────────────────
print("2️⃣  Downloading Faster Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster Whisper model download failed: {e}\n")

# ── 3. Qwen TTS model ───────────────────────────────────────────
print("3️⃣  Downloading Qwen TTS model...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    print("   ✅ Qwen TTS model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen TTS download failed: {e}\n")

# ── 4. Qwen VoiceDesign model (for voice themes) ────────────────
print("4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
    print("   ✅ Qwen VoiceDesign model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen VoiceDesign download failed: {e}\n")

# ── 5. Warm up Ollama (load model into GPU) ──────────────────────
print("5️⃣  Warming up Ollama LLM (loading model into GPU)...")
if _ollama_ok:
    try:
        import json, urllib.request
        _body = json.dumps({
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": "Reply with: ready"}],
            "stream": False,
            "think": False,
            "options": {"temperature": 0.1},
        }).encode()
        _req = urllib.request.Request(
            "http://localhost:11434/api/chat", _body,
            headers={"Content-Type": "application/json"},
        )
        t0 = time.time()
        with urllib.request.urlopen(_req, timeout=120) as _resp:
            _result = json.loads(_resp.read())
        _reply = _result.get("message", {}).get("content", "")
        _tokens = _result.get("eval_count", 0)
        print(f"   ✅ Ollama responded in {time.time() - t0:.1f}s ({_tokens} tokens): {_reply[:50]}\n")
    except Exception as e:
        print(f"   ⚠️  Warm-up failed: {e}\n")
else:
    print("   ⏭️  Skipped (model not available)\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

# ── Benchmark Ollama LLM with test image ─────────────────────────
if _ollama_ok:
    try:
        from PIL import Image as _PILImage
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow"])
        from PIL import Image as _PILImage

    _img = _PILImage.new("RGB", (64, 64), color=(200, 60, 60))
    _buf = io.BytesIO()
    _img.save(_buf, format="JPEG", quality=70)
    _b64 = base64.b64encode(_buf.getvalue()).decode()

    # ── Build mazinger LLM client (native Ollama, thinking disabled) ─
    from mazinger.llm import build_client

    _client = build_client(
        base_url="http://localhost:11434/v1",
        think=False,
    )

    _model = OLLAMA_MODEL
    _messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in one sentence."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{_b64}"}},
            ],
        }
    ]

    # ── Run benchmark ────────────────────────────────────────────────
    print(f"🏎️  Benchmarking {_model} with a 64×64 test image…\n")

    try:
        _t0 = time.time()
        _resp = _client.chat.completions.create(model=_model, messages=_messages, temperature=0.1)
        _elapsed = time.time() - _t0

        _text = _resp.choices[0].message.content
        _prompt_tok = _resp.usage.prompt_tokens
        _gen_tok = _resp.usage.completion_tokens
        _tok_per_sec = _gen_tok / _elapsed if _elapsed > 0 else 0

        print(f"   Response   : {_text[:120]}")
        print(f"   Prompt tok : {_prompt_tok}")
        print(f"   Gen tokens : {_gen_tok}")
        print(f"   Wall time  : {_elapsed:.2f}s")
        print(f"   Speed      : {_tok_per_sec:.1f} tok/s")
        print()

        if _tok_per_sec < 5:
            print("⚠️  Throughput is low — consider a smaller model or check GPU availability.")
        elif _tok_per_sec < 20:
            print("✅ Reasonable speed. Dubbing will work but may be a bit slow on long videos.")
        else:
            print("🚀 Great throughput! You're ready to dub.")
    except Exception as e:
        print(f"   ⚠️  Benchmark failed: {e}")
        print("   The model may not support vision. Dubbing will still work for text-only tasks.")
else:
    print("\n⏭️  Skipping benchmark (Ollama model not available).")
    print(f"   Try pulling the model manually: !ollama pull {OLLAMA_MODEL}")

In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import os, sys, urllib.request

# ── Ensure OLLAMA_MODEL env var is set (in case Step 2 was skipped) ──
os.environ.setdefault("OLLAMA_MODEL", "qwen3.5:2b-bf16")

# ── Detect environment (re-detect in case earlier cells were skipped) ─
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# ── Fetch studio scripts from GitHub ─────────────────────────────
_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/notebooks/studio"
_SCRIPTS = ["constants.py", "theme.py", "helpers.py", "pipeline.py", "app.py"]

os.makedirs("studio", exist_ok=True)
for _script in _SCRIPTS:
    _dest = os.path.join("studio", _script)
    if not os.path.exists(_dest):
        urllib.request.urlretrieve(f"{_BASE}/{_script}", _dest)
        print(f"  ✅ Downloaded {_script}")
    else:
        print(f"  ✔ {_script} (cached)")

print(f"\n🚀 Launching Mazinger Studio (Ollama model: {os.environ['OLLAMA_MODEL']})...\n")

# ── Run the app inline so Gradio can display its link in the notebook ─
sys.path.insert(0, "studio")
from app import app

app.launch(share=True, debug=True, show_error=True, inbrowser=False)